# Wordle: Human vs Greedy Hill Climbing

This notebook implements:

1. A Wordle game engine.
2. An interactive manual game loop for a human player.
3. A greedy hill-climbing solver for comparison.

## Data source

- The notebook reads words from `word_list.txt` (one 5-letter word per line).
- If `word_list.txt` is not present, it falls back to `words.txt`.
- The same list is used for both valid guesses and secret answers.

## Emoji feedback

- 🟩: correct letter in the correct position (green)
- 🟨: correct letter in the wrong position (yellow)
- ⬜: letter not in the word (grey)

In [5]:
from pathlib import Path
from collections import Counter
import random


def load_word_list(source):
    words = [
        line.strip().lower()
        for line in source.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    words = [w for w in words if len(w) == 5 and w.isalpha()]

    if not words:
        raise ValueError("Word list is empty after filtering for 5-letter alphabetic words.")

    # Keep order stable but remove duplicates.
    words = list(dict.fromkeys(words))
    return words, source


WORDS, WORD_SOURCE = load_word_list(Path("words.txt"))
ALLOWED_WORDS, ALLOWED_SOURCE = load_word_list(Path("allowed.txt"))
MAX_TURNS = 6

print(f"Loaded {len(WORDS)} words from {WORD_SOURCE.name}")

Loaded 2315 words from words.txt


## Wordle Game Engine

The feedback engine returns one status per letter:

- `G` for green
- `Y` for yellow
- `B` for grey

The algorithm handles duplicate letters correctly by assigning greens first, then yellows from remaining counts.

In [6]:
EMOJI_MAP = {"G": "🟩", "Y": "🟨", "B": "⬜"}


def wordle_feedback(secret, guess):
    """Return tuple feedback using G/Y/B symbols with duplicate-letter correctness."""
    secret = secret.lower()
    guess = guess.lower()

    feedback = ["B"] * 5
    remaining = Counter()

    # Pass 1: mark greens and count remaining secret letters.
    for i, (s, g) in enumerate(zip(secret, guess)):
        if g == s:
            feedback[i] = "G"
        else:
            remaining[s] += 1

    # Pass 2: mark yellows using remaining counts.
    for i, g in enumerate(guess):
        if feedback[i] == "G":
            continue
        if remaining[g] > 0:
            feedback[i] = "Y"
            remaining[g] -= 1

    return tuple(feedback)


def feedback_to_emoji(feedback):
    return "".join(EMOJI_MAP[c] for c in feedback)


def format_guess_line(guess, feedback):
    return f"{guess.upper()} {feedback_to_emoji(feedback)}"


def validate_guess(guess, words):
    g = guess.strip().lower()
    if len(g) != 5 or not g.isalpha():
        return False, "Guess must be exactly 5 letters."
    if g not in words:
        return False, "Guess is not in the word list."
    return True, g

In [7]:
# Quick Counter demo for duplicate-letter behavior.
example_secret = "apple"
example_guess = "allee"
example_feedback = wordle_feedback(example_secret, example_guess)
print(f"Secret: {example_secret.upper()} | Guess: {example_guess.upper()}")
print(format_guess_line(example_guess, example_feedback))
print("Legend: G=🟩, Y=🟨, B=⬜")

Secret: APPLE | Guess: ALLEE
ALLEE 🟩🟨⬜⬜🟩
Legend: G=🟩, Y=🟨, B=⬜


## Why Counter Is Used (and How Colors Are Assigned)

`Counter` from `collections` counts how many times each letter appears.

In Wordle feedback with duplicates, we must avoid over-counting yellow letters:

1. Mark all greens first (`G` / 🟩).
2. Count only the remaining unmatched letters in the secret using `Counter`.
3. For each non-green letter in the guess:
   - If that letter still has count > 0, mark yellow (`Y` / 🟨) and decrease the count.
   - Otherwise mark grey (`B` / ⬜).

That is exactly why `Counter` is the safest way to make Wordle colors correct.

## Manual Mode

Run the next cell to play Wordle yourself.

For each guess, the notebook prints:

`GUESS` + emoji feedback, for example `POWER 🟩⬜🟨⬜🟩`

In [8]:
def play_wordle_manual(secret, allowed_words, max_turns=MAX_TURNS):
    """Interactive Wordle loop for a human player."""
    history = []

    print("\nWordle manual mode started.")
    print(f"You have {max_turns} turns. Enter 5-letter guesses from the word list.")

    for turn in range(1, max_turns + 1):
        while True:
            raw = input(f"Turn {turn}/{max_turns} guess: ")
            ok, value = validate_guess(raw, allowed_words)
            if ok:
                guess = value
                break
            print(f"Invalid guess: {value}")

        feedback = wordle_feedback(secret, guess)
        history.append((guess, feedback))
        print(format_guess_line(guess, feedback))

        if guess == secret:
            print(f"Solved in {turn} turns.")
            return {
                "solved": True,
                "turns": turn,
                "history": history,
                "secret": secret,
            }

    print("Not solved within turn limit.")
    return {
        "solved": False,
        "turns": max_turns,
        "history": history,
        "secret": secret,
    }

## Greedy Hill Climbing Solver

This solver follows your required logic exactly:

- Step A: start with full candidate list.
- Step B: recalculate letter frequencies from candidates only.
- Step C: score each candidate using unique-letter frequency sum.
- Step D: pick highest score, alphabetical tie-break.
- Step E: prune candidates using observed feedback.
- Step F: repeat until solved.

In [9]:
def compute_letter_frequencies(candidates):
    """Step B: frequency table from current candidate list only."""
    freq = Counter()
    for w in candidates:
        freq.update(w)
    return freq


def score_candidate(word, letter_freq):
    """Step C: sum frequencies of unique letters (no double-counting)."""
    return sum(letter_freq[ch] for ch in set(word))


def choose_greedy_guess(candidates):
    """Step D: max score; alphabetical tie-break."""
    letter_freq = compute_letter_frequencies(candidates)

    # Sort by (-score, word) so ties are broken alphabetically.
    ranked = sorted(
        candidates,
        key=lambda w: (-score_candidate(w, letter_freq), w),
    )
    best = ranked[0]
    best_score = score_candidate(best, letter_freq)
    return best, best_score, letter_freq


def prune_candidates(candidates, guess, observed_feedback):
    """Step E: keep words whose feedback vs guess matches observed feedback."""
    return [
        w for w in candidates if wordle_feedback(w, guess) == observed_feedback
    ]


def solve_wordle_greedy_hill(secret, words, max_turns=MAX_TURNS, verbose=True):
    """Greedy hill climbing Wordle solver implementing Steps A-F."""
    # Step A: Start with the full word list as candidates.
    candidates = list(words)
    history = []

    if verbose:
        print(f"Step A: start with {len(candidates)} candidates")

    for turn in range(1, max_turns + 1):
        # Steps B, C, D happen inside this guess selection call.
        guess, guess_score, letter_freq = choose_greedy_guess(candidates)

        feedback = wordle_feedback(secret, guess)
        history.append((guess, feedback))

        if verbose:
            unique_letters = "".join(sorted(set(guess)))
            top_letters = sorted(letter_freq.items(), key=lambda kv: (-kv[1], kv[0]))[:5]
            top_letters_text = ", ".join(f"{ch}:{count}" for ch, count in top_letters)
            print(f"\nTurn {turn}")
            print(f"Step B: recomputed frequencies from {len(candidates)} candidates")
            print(f"Step C: unique letters in guess: {unique_letters}")
            print(f"Step C: score({guess.upper()}) = {guess_score}")
            print(f"Step D: chose best guess with alphabetical tie-break")
            print(f"Top letters this turn: {top_letters_text}")
            print(format_guess_line(guess, feedback))
            print("Colors: 🟩=green, 🟨=yellow, ⬜=grey")

        if guess == secret:
            if verbose:
                print(f"Solved in {turn} turns.")
            return {
                "solved": True,
                "turns": turn,
                "history": history,
                "secret": secret,
            }

        # Step E: prune candidates, then Step F repeats loop.
        before = len(candidates)
        candidates = prune_candidates(candidates, guess, feedback)

        if verbose:
            print(f"Step E: pruned candidates {before} -> {len(candidates)}")
            print("Step F: repeat with remaining candidates")

        if not candidates:
            break

    if verbose:
        print("Greedy hill climber did not solve within turn limit.")
    return {
        "solved": False,
        "turns": max_turns,
        "history": history,
        "secret": secret,
    }

## Human vs AI Comparison

Run the next cell:

1. A random secret word is chosen from the same pool.
2. You play first in manual mode.
3. The greedy hill climber solves the exact same secret.
4. The notebook prints a turn comparison summary.

In [10]:
secret_word = random.choice(WORDS)

print("New playthrough started.")
print("A random secret word was selected. Human plays first.\n")

human_result = play_wordle_manual(secret_word, ALLOWED_WORDS, max_turns=MAX_TURNS)

print("\nNow running greedy hill climber on the same secret...\n")
ai_result = solve_wordle_greedy_hill(secret_word, WORDS, max_turns=MAX_TURNS, verbose=True)

print("\n===== Summary =====")
print(f"Secret word: {secret_word.upper()}")
print(f"Human solved: {human_result['solved']} | Turns: {human_result['turns']}")
print(f"AI solved:    {ai_result['solved']} | Turns: {ai_result['turns']}")

if human_result["solved"] and ai_result["solved"]:
    if human_result["turns"] < ai_result["turns"]:
        print("Winner: Human")
    elif human_result["turns"] > ai_result["turns"]:
        print("Winner: Greedy Hill Climber")
    else:
        print("Result: Tie")
elif human_result["solved"] and not ai_result["solved"]:
    print("Winner: Human (AI did not solve in time)")
elif ai_result["solved"] and not human_result["solved"]:
    print("Winner: Greedy Hill Climber (human did not solve in time)")
else:
    print("Result: Neither solved within turn limit")

New playthrough started.
A random secret word was selected. Human plays first.


Wordle manual mode started.
You have 6 turns. Enter 5-letter guesses from the word list.
SALET ⬜🟨🟨🟨⬜
GNARL ⬜⬜🟨⬜🟩
MEDAL 🟩🟩🟩🟩🟩
Solved in 3 turns.

Now running greedy hill climber on the same secret...

Step A: start with 2315 candidates

Turn 1
Step B: recomputed frequencies from 2315 candidates
Step C: unique letters in guess: aelrt
Step C: score(ALERT) = 4559
Step D: chose best guess with alphabetical tie-break
Top letters this turn: e:1233, a:979, r:899, o:754, t:729
ALERT 🟨🟨🟨⬜⬜
Colors: 🟩=green, 🟨=yellow, ⬜=grey
Step E: pruned candidates 2315 -> 42
Step F: repeat with remaining candidates

Turn 2
Step B: recomputed frequencies from 42 candidates
Step C: unique letters in guess: aelsv
Step C: score(SALVE) = 151
Step D: chose best guess with alphabetical tie-break
Top letters this turn: l:47, e:46, a:42, s:8, v:8
SALVE ⬜🟨🟨⬜🟨
Colors: 🟩=green, 🟨=yellow, ⬜=grey
Step E: pruned candidates 42 -> 12
Step F: repeat